In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIXED #####
# changed animal_shelter and AnimalShelter to match yCRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIXED updated username and password and CRUD Python module name

username = "aacuser"
password = "110825"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIXED added in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # FIXED: Correct name and path
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIXED Placed the HTML image tag in the line below into the app.layout code according to your design
#FIXED included a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '80px'}
        ),
        html.A("Grazioso Salvare", href="https://www.snhu.edu"), #FIXED: Company requested logo include a URL anchor tag to the client’s home page: www.snhu.edu.
    ], style={'textAlign': 'center'}),
    html.Center(html.B(html.H2('CS-340 Dashboard for Erin Swinson'))),
    html.Hr(),
    html.Div(
        
#FIXED Added in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.

    id='filter-div',
    children=[
        html.H4("Rescue Type Filter"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'} #FIXED: Chnaged style to match example
        )
    ]
),
    html.Hr(),
#FIXED Set up the features for interactive data table to make it user-friendly for your client
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'},
    ),

    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#### Created a query function ####
def build_query(filter_type):
    if filter_type == 'water':
        return {
            "animal_type": "Dog",
            "age_upon_outcome_in_weeks": {"$lte": 104},
            "breed": {
                "$in": [
                    "Labrador Retriever Mix", "Chesapeake Bay Retriever",
                    "Newfoundland", "Golden Retriever Mix"
                ]
            }
        }
    elif filter_type == 'mountain':
        return {
            "animal_type": "Dog",
            "age_upon_outcome_in_weeks": {"$lte": 156},
            "breed": {
                "$in": [
                    "German Shepherd", "Border Collie", "Australian Shepherd",
                    "Bernese Mountain Dog"
                ]
            }
        }
    elif filter_type == 'disaster':
        return {
            "animal_type": "Dog",
            "age_upon_outcome_in_weeks": {"$lte": 156},
            "breed": {
                "$in": [
                    "German Shepherd", "Belgian Malinois", "Bloodhound",
                    "Doberman Pinscher"
                ]
            }
        }
    else:  # reset
        return {}

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIXED Added code to filter interactive data table with MongoDB queries      
    query = build_query(filter_type)
    records = list(db.read(query))
    if not records:
        return []
    dff = pd.DataFrame.from_records(records)
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ### FIXED Implemented graph for pie chart
    if viewData is None or len(viewData) == 0:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    fig = px.pie(
        dff,
        names='breed',
        title='Breed Distribution for Selected Rescue Filter'
    )
    
    #FIXED: too many results causes the pie chart to distort
    fig.update_traces(textposition='inside', textinfo='percent+label', hole=0)
    fig.update_layout(
        margin=dict(t=50, b=50, l=50, r=50),
        uniformtext_minsize=12,
        uniformtext_mode='hide'
    )
    
    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns] #Improved safegaurds for None


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])

#FIXED list of children for map-id, not "None"
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ])
            ])
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://carolgreen-preludemercy-3000.codio.io/proxy/8050/
